In [4]:
import os
import zstandard  # pip install zstandard
from tqdm import tqdm
import random

In [5]:
def zst_files_in_dir(directory):
    """List all .zst files in a directory."""
    files = []
    for filename in os.listdir(directory):
        if filename.endswith(".zst") and os.path.isfile(os.path.join(directory, filename)):
            files.append(filename)
    return files

In [6]:
def decompress_zst_to_text(input_file, output_file, vocab):
    """Decompresses a .zst file and extracts text to output_file, updating vocab."""
    with open(input_file, "rb") as infile:
        dctx = zstandard.ZstdDecompressor()
        with dctx.stream_reader(infile) as reader:
            current_line = b""
            while True:
                chunk = reader.read(16384)  # Read in 16KB chunks
                if not chunk:
                    break
                current_line += chunk
                # Split into lines (handles partial lines)
                lines = current_line.split(b"\n")
                current_line = lines.pop()
                # Process each line
                for line in lines:
                    try:
                        text = line.decode("utf-8").strip()
                        if text:
                            yield text
                            characters = set(text)
                            vocab.update(characters)
                    except UnicodeDecodeError:
                        pass  # Skip invalid UTF-8 sequences
            # Process the remaining partial line
            if current_line:
                try:
                    text = current_line.decode("utf-8").strip()
                    if text:
                        yield text
                        characters = set(text)
                        vocab.update(characters)
                except UnicodeDecodeError:
                    pass

In [7]:
folder_path = "openwebtext2"
output_file_train = "output_train_v3.txt"
output_file_val = "output_val_v3.txt"
vocab_file = "vocab_v3.txt"


In [8]:
# Gather files
files = zst_files_in_dir(folder_path)
total_files = len(files)
print(f"Total files: {total_files}")
print(files)

Total files: 179
['2005-06.jsonl.zst', '2005-07.jsonl.zst', '2005-08.jsonl.zst', '2005-09.jsonl.zst', '2005-10.jsonl.zst', '2005-11.jsonl.zst', '2005-12.jsonl.zst', '2006-01.jsonl.zst', '2006-02.jsonl.zst', '2006-03.jsonl.zst', '2006-04.jsonl.zst', '2006-05.jsonl.zst', '2006-06.jsonl.zst', '2006-07.jsonl.zst', '2006-08.jsonl.zst', '2006-09.jsonl.zst', '2006-10.jsonl.zst', '2006-11.jsonl.zst', '2006-12.jsonl.zst', '2007-01.jsonl.zst', '2007-02.jsonl.zst', '2007-03.jsonl.zst', '2007-04.jsonl.zst', '2007-05.jsonl.zst', '2007-06.jsonl.zst', '2007-07.jsonl.zst', '2007-08.jsonl.zst', '2007-09.jsonl.zst', '2007-10.jsonl.zst', '2007-11.jsonl.zst', '2007-12.jsonl.zst', '2008-01.jsonl.zst', '2008-02.jsonl.zst', '2008-03.jsonl.zst', '2008-04.jsonl.zst', '2008-05.jsonl.zst', '2008-06.jsonl.zst', '2008-07.jsonl.zst', '2008-08.jsonl.zst', '2008-09.jsonl.zst', '2008-10.jsonl.zst', '2008-11.jsonl.zst', '2008-12.jsonl.zst', '2009-01.jsonl.zst', '2009-02.jsonl.zst', '2009-03.jsonl.zst', '2009-04.jsonl.z

In [9]:
# Shuffle files randomly 
random.seed(42)  # Optional: Set seed for reproducibility
random.shuffle(files)  # Shuffle in-place
print(files)

['2018-02.jsonl.zst', '2009-11.jsonl.zst', '2006-09.jsonl.zst', '2008-12.jsonl.zst', '2006-07.jsonl.zst', '2008-06.jsonl.zst', '2010-06.jsonl.zst', '2015-08.jsonl.zst', '2010-07.jsonl.zst', '2007-01.jsonl.zst', '2010-11.jsonl.zst', '2007-12.jsonl.zst', '2018-05.jsonl.zst', '2005-08.jsonl.zst', '2016-08.jsonl.zst', '2016-09.jsonl.zst', '2015-06.jsonl.zst', '2018-09.jsonl.zst', '2019-07.jsonl.zst', '2010-12.jsonl.zst', '2013-04.jsonl.zst', '2019-11.jsonl.zst', '2007-07.jsonl.zst', '2016-06.jsonl.zst', '2017-10.jsonl.zst', '2018-08.jsonl.zst', '2019-12.jsonl.zst', '2011-08.jsonl.zst', '2017-03.jsonl.zst', '2017-07.jsonl.zst', '2006-08.jsonl.zst', '2009-10.jsonl.zst', '2016-02.jsonl.zst', '2014-02.jsonl.zst', '2009-02.jsonl.zst', '2015-09.jsonl.zst', '2011-03.jsonl.zst', '2012-02.jsonl.zst', '2018-10.jsonl.zst', '2005-06.jsonl.zst', '2015-01.jsonl.zst', '2006-10.jsonl.zst', '2016-10.jsonl.zst', '2015-11.jsonl.zst', '2013-10.jsonl.zst', '2010-10.jsonl.zst', '2018-06.jsonl.zst', '2012-05.jso

In [10]:
# Split files into train/val (90%/10%)
split_index = int(total_files * 0.9)
files_train = files[:split_index]
files_val = files[split_index:]
vocab = set()

In [11]:
# Process training files
with open(output_file_train, "w", encoding="utf-8") as outf:
    for filename in tqdm(files_train, total=len(files_train), desc="Processing Train"):
        file_path = os.path.join(folder_path, filename)
        try:
            for text in decompress_zst_to_text(file_path, outf.name, vocab):
                outf.write(text + "\n")
        except Exception as e:
            print(f"Error processing {file_path}: {e}")

Processing Train: 100%|██████████| 161/161 [22:22<00:00,  8.34s/it]


In [12]:
# Process validation files
with open(output_file_val, "w", encoding="utf-8") as outf:
    for filename in tqdm(files_val, total=len(files_val), desc="Processing Val"):
        file_path = os.path.join(folder_path, filename)
        try:
            for text in decompress_zst_to_text(file_path, outf.name, vocab):
                outf.write(text + "\n")
        except Exception as e:
            print(f"Error processing {file_path}: {e}")


Processing Val: 100%|██████████| 18/18 [01:20<00:00,  4.50s/it]


In [13]:
# Write vocabulary
with open(vocab_file, "w", encoding="utf-8") as vfile:
    for char in sorted(vocab):
        vfile.write(char + "\n")
        